In [1]:
import os
import pandas as pd

# List files in the current working directory
print("Current directory:", os.getcwd())
print("\nAvailable files:")

for file in os.listdir():
    print(file)



df = pd.read_csv("../data/Sample - Superstore.csv", encoding="cp1252")

print("Dataset loaded successfully")
print("Shape:", df.shape)

display(df.head())

Current directory: d:\consulting\Retail Profitability Turnaround\notebook

Available files:
01_load_data.ipynb
02_analyis.ipynb
03_Root-Cause Analysis of Profitability Gaps.ipynb
04_Product_Region_Profitability.ipynb
05_Discount_Product_Analysis.ipynb
Dataset loaded successfully
Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [2]:
target_products = df[
    df["Sub-Category"].isin(["Tables", "Bookcases"])
].copy()

discount_product_analysis = (
    target_products
    .groupby(["Sub-Category", "Discount"])
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Records=("Profit", "size"),
        Loss_Making_Records=("Profit", lambda x: (x < 0).sum())
    )
    .reset_index()
)

discount_product_analysis["Profit_Margin_%"] = (
    discount_product_analysis["Profit"]
    / discount_product_analysis["Sales"] * 100
)

discount_product_analysis["Loss_Record_Share_%"] = (
    discount_product_analysis["Loss_Making_Records"]
    / discount_product_analysis["Records"] * 100
)

display(
    discount_product_analysis
    .sort_values(["Sub-Category", "Discount"])
    .round(2)
)

,Sub-Category,Discount,Sales,Profit,Records,Loss_Making_Records,Profit_Margin_%,Loss_Record_Share_%
0,Bookcases,0.00,31935.98,6075.71,60,0,19.02,0.00
1,Bookcases,0.15,27558.52,1418.99,52,17,5.15,32.69
2,Bookcases,0.20,26841.50,130.50,46,23,0.49,50.00
3,Bookcases,0.30,4282.70,-555.87,10,9,-12.98,90.00
4,Bookcases,0.32,14493.46,-2391.14,27,27,-16.50,100.00
5,Bookcases,0.50,7308.46,-4255.81,18,18,-58.23,100.00
6,Bookcases,0.70,2459.38,-3894.94,15,15,-158.37,100.00
7,Tables,0.00,71578.76,13276.30,72,0,18.55,0.00
8,Tables,0.20,45430.23,-303.56,71,31,-0.67,43.66
9,Tables,0.30,25182.15,-3402.33,54,50,-13.51,92.59


Objective: Examine how discount levels relate to profitability in Tables and Bookcases.

Key findings

Tables become unprofitable at a lower observed discount level. At a 20% discount, Tables recorded a loss of $303.56 and a profit margin of −0.67%. At a 30% discount, the loss increased to $3,402.33, with 92.59% of records being loss-making.

Bookcases approach break-even at a 20% discount. Their aggregate profit at this level was $130.50, with a margin of 0.49%. At a 30% discount, the subcategory recorded a loss of $555.87, and 90% of records were loss-making.

High-discount groups show severe negative profitability. Every observed Table record at discounts of 40% or more was loss-making. For Bookcases, every observed record at discounts of 32%, 50%, and 70% was loss-making.

Business implication: Discount practices warrant further investigation, with particular attention to Tables. However, these descriptive results do not establish that discounts alone caused the losses. The observed discount levels should not automatically be treated as optimal pricing thresholds.